<a href="https://colab.research.google.com/github/Albedofan69420/SIMPSONS-CNN/blob/main/Evaluaci%C3%B3n2_deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Integrantes

*   Soledad Faúndez



#Comprensión del caso

Se comprende la poseción de un dataset llamado "The Simpsons Characters Data", el cual contiene más de 20.000 imágenes de 42 personajes diferentes de la famosa serie animada "Los Simpsons", los cuales se contienen dentro de carpetas identificadas con los nombres de cada uno.

##Objetivos del modelo

1.- Desarrollar un modelo con CNN para identificar patrones a partir de las imágenes de cada uno de los personajes de la serie, con el propósito de clasificar nuevas imágenes de forma correctamente.

2.- Realizar variadas pruebas en base a diferentes hiperparametros y comparando modelos para lograr identificar el mejor modelo para clasificar imágenes de los Simpsons.

#Carga y preprocesamiento de datos

In [ ]:
!pip install gdown

In [ ]:
#Se importan las librerias necesarias para poder trabajar a lo largo del documento

from google.colab.patches import cv2_imshow
import cv2
import glob
%matplotlib inline

import tensorflow as tf

from keras.models import Sequential
from keras.layers import Dense, Activation , Dropout, Flatten, BatchNormalization, Conv2D, MaxPooling2D
from keras.src.legacy.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import collections
from keras.optimizers import Adam, SGD
from sklearn.metrics import confusion_matrix
from sklearn import metrics
import itertools

In [ ]:
#Se descargan los sets de datos necesarios (train y test respectivamente) y los descomprime
file_id = '1B-RXOFSXiPHva-E_X5yPFlore_zlN-4d'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.tar.gz'

gdown.download(url, output, quiet=True)
!tar -xzvf data.tar.gz

second_file_id = '1OwJI6tQSu90cbo91yCI43JK6-Zh22Cpp'
second_url = f'https://drive.google.com/uc?id={second_file_id}'
second_output = 'second_data.tar.gz'

gdown.download(second_url, second_output, quiet=True)
!tar -xzvf second_data.tar.gz

Se han truncado las últimas 5000 líneas del flujo de salida.
simpsons/mayor_quimby/pic_0116.jpg
simpsons/milhouse_van_houten/pic_0576.jpg
simpsons/lenny_leonard/pic_0149.jpg
simpsons/kent_brockman/pic_0446.jpg
simpsons/nelson_muntz/pic_0060.jpg
simpsons/krusty_the_clown/pic_0838.jpg
simpsons/homer_simpson/pic_0637.jpg
simpsons/homer_simpson/pic_0495.jpg
simpsons/krusty_the_clown/pic_0893.jpg
simpsons/homer_simpson/pic_0834.jpg
simpsons/homer_simpson/pic_0692.jpg
simpsons/lisa_simpson/pic_0755.jpg
simpsons/marge_simpson/pic_0654.jpg
simpsons/chief_wiggum/pic_0344.jpg
simpsons/lisa_simpson/pic_0952.jpg
simpsons/lisa_simpson/pic_1269.jpg
simpsons/marge_simpson/pic_0851.jpg
simpsons/marge_simpson/pic_1168.jpg
simpsons/chief_wiggum/pic_0541.jpg
simpsons/homer_simpson/pic_1948.jpg
simpsons/sideshow_bob/pic_0104.jpg
simpsons/lisa_simpson/pic_0278.jpg
simpsons/nelson_muntz/pic_0128.jpg
simpsons/marge_simpson/pic_0177.jpg
simpsons/milhouse_van_houten/pic_0699.jpg
simpsons/waylon_smithers/pic_00

In [ ]:
# Esta variable contiene un mapeo de número de clase a personaje.
# Utilizamos sólo los 18 personajes del dataset que tienen más imágenes.
MAP_CHARACTERS = {
    0: 'abraham_grampa_simpson', 1: 'apu_nahasapeemapetilon', 2: 'bart_simpson',
    3: 'charles_montgomery_burns', 4: 'chief_wiggum', 5: 'comic_book_guy', 6: 'edna_krabappel',
    7: 'homer_simpson', 8: 'kent_brockman', 9: 'krusty_the_clown', 10: 'lisa_simpson',
    11: 'marge_simpson', 12: 'milhouse_van_houten', 13: 'moe_szyslak',
    14: 'ned_flanders', 15: 'nelson_muntz', 16: 'principal_skinner', 17: 'sideshow_bob'
}

# Vamos a estandarizar todas las imágenes a tamaño 64x64
IMG_SIZE = 64

In [ ]:
#Carga los datos de training en imágenes etiquetadas
def load_train_set(dirname, map_characters, verbose=True):
    X_train = []
    y_train = []
    for label, character in map_characters.items():
        files = os.listdir(os.path.join(dirname, character))
        images = [file for file in files if file.endswith("jpg")]
        if verbose:
          print("Leyendo {} imágenes encontradas de {}".format(len(images), character))
        for image_name in images:
            image = cv2.imread(os.path.join(dirname, character, image_name))
            X_train.append(cv2.resize(image,(IMG_SIZE, IMG_SIZE)))
            y_train.append(label)
    return np.array(X_train), np.array(y_train)

In [ ]:
#Esta función funciona de manera equivalente a la función load_train_set pero cargando los datos de test. Extrae los nombres de las fotos y etiqueta
def load_test_set(dirname, map_characters, verbose=True):
    X_test = []
    y_test = []
    reverse_dict = {v: k for k, v in map_characters.items()}
    for filename in glob.glob(dirname + '/*.*'):
        char_name = "_".join(filename.split('/')[-1].split('_')[:-1])
        if char_name in reverse_dict:
            image = cv2.imread(filename)
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
            X_test.append(image)
            y_test.append(reverse_dict[char_name])
    if verbose:
        print("Leídas {} imágenes de test".format(len(X_test)))
    return np.array(X_test), np.array(y_test)

In [ ]:
# Cargamos los datos
DATASET_TRAIN_PATH_COLAB = "simpsons"
DATASET_TEST_PATH_COLAB = "simpsons_testset"

X, y = load_train_set(DATASET_TRAIN_PATH_COLAB, MAP_CHARACTERS)
X_t, y_t = load_test_set(DATASET_TEST_PATH_COLAB, MAP_CHARACTERS)

Leyendo 913 imágenes encontradas de abraham_grampa_simpson
Leyendo 623 imágenes encontradas de apu_nahasapeemapetilon
Leyendo 1342 imágenes encontradas de bart_simpson
Leyendo 1193 imágenes encontradas de charles_montgomery_burns
Leyendo 986 imágenes encontradas de chief_wiggum
Leyendo 469 imágenes encontradas de comic_book_guy
Leyendo 457 imágenes encontradas de edna_krabappel
Leyendo 2246 imágenes encontradas de homer_simpson
Leyendo 498 imágenes encontradas de kent_brockman
Leyendo 1206 imágenes encontradas de krusty_the_clown
Leyendo 1354 imágenes encontradas de lisa_simpson
Leyendo 1291 imágenes encontradas de marge_simpson
Leyendo 1079 imágenes encontradas de milhouse_van_houten
Leyendo 1452 imágenes encontradas de moe_szyslak
Leyendo 1454 imágenes encontradas de ned_flanders
Leyendo 358 imágenes encontradas de nelson_muntz
Leyendo 1194 imágenes encontradas de principal_skinner
Leyendo 877 imágenes encontradas de sideshow_bob
Leídas 890 imágenes de test


In [ ]:
# Vamos a barajar aleatoriamente los datos. Esto es importante ya que si no
# lo hacemos y, por ejemplo, cogemos el 20% de los datos finales como validation
# set, estaremos utilizando solo un pequeño número de personajes, ya que
# las imágenes se leen secuencialmente personaje a personaje.
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]

In [ ]:
#Normalización de datos
X_train = X / 255.0
X_test = X_t / 255.0

Y_train = y
Y_test = y_t

#Verificación de dimensiones. Se ven representados la cantidad de datos, resolución y RGB
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)

Dentro de la etapa de preprocesamiento de datos se hicieron los siguientes cambios:

*   
*   Elemento de lista



#Definición del modelo CNN

In [ ]:
def create_model():
    model = Sequential([

        Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Flatten(),

        Dense(128, activation='relu'),

        Dropout(0.5),

        Dense(len(MAP_CHARACTERS), activation='softmax')
    ])

    model.compile(
      optimizer='adam',
      loss='sparse_categorical_crossentropy',
      metrics=['accuracy']
    )

    return model

#Entrenamiento y ajuste de hiperparametros